#Phase 2: Aspect Taxonomy
-  Step 2.1: Representative Stratified Sampling
-  Step 2.2: Generative Semantic Analysis
-  Step 2.3: Global Taxonomy Formalization and Validation


In [6]:
import os
import pandas as pd

base_dir = 'D:/Ateneo de Davao/Thesis-Project-Aspect-Based-Sentiment-Analysis/Dataset'
file_path = os.path.join(base_dir, 'Thesis_master_reviews_combined.csv')

df = pd.read_csv(file_path)

#remove nulls since it wouldnt be necessary now

In [7]:
# 1. Clean dirty data immediately
df = df.dropna(subset=['clean_review'])
df = df[df['clean_review'].astype(str).str.strip() != '']

# Enforce lowercase formatting
df.columns = df.columns.str.lower()
df['shop_type'] = df['shop_type'].astype(str).str.lower().str.strip()

# Save the cleaned master dataset
clean_master_path = os.path.join(base_dir, 'Thesis_Master_reviews_nonull.csv')
df.to_csv(clean_master_path, index=False, encoding='utf-8-sig')
print(f"Cleaned master dataset saved to: {clean_master_path}")

Cleaned master dataset saved to: D:/Ateneo de Davao/Thesis-Project-Aspect-Based-Sentiment-Analysis/Dataset\Thesis_Master_reviews_nonull.csv


#Step 2.1: Representative Stratified Sampling

In [8]:
base_dir = 'D:/Ateneo de Davao/Thesis-Project-Aspect-Based-Sentiment-Analysis/Dataset'
#^^^
file_path = os.path.join(base_dir, 'Thesis_Master_reviews_nonull.csv') 

df = pd.read_csv(file_path)

In [9]:
local_markers = [
    # Bisaya/Cebuano
    'ang', 'lami', 'kaayo', 'ilang', 'nalang', 'gyud', 'dili', 'dakong', 
    'diri', 'imo', 'walay', 'daghan', 'murag', 'inyo', 'kani', 'langaw', 'pila', 
    'umay', 'ilahang', 'mubalik', 'kauban', 'layo', 'taas', 'asin', 'tibuok', 'lamiay', 'muni',
    
    # Tagalog
    'yung', 'nila', 'naman', 'talaga', 'kaya', 'kasi', 'siya', 'sana', 'medyo', 'pwd', 
    'sya', 'dito', 'pag', 'walang', 'suki', 'sobrang', 'lng', 'sayang', 'isa', 'unli', 
    'pwede', 'mabait', 'meron', 'tlga', 'pagka', 'tipid', 'nung', 'dami', 'chika', 'ano', 
    'tita', 'medj', 'silang', 'isang', 'araw', 'babalik', 'malayo', 'alam', 'niyo', 
    'maraming', 'kase', 'parin', 'matagal', 'niya', 'ninyo', 'pwds', 'nlng', 'tropa', 'huli', 
    'maayos', 'sige', 'ibalik', 'muni-muni',
    
    # Shared words Tagalog and Bisaya/Cebuano
    'wala', 'masarap', 'mga', 'sila', 'nga', 'grabe', 'maayo', 'kasama'
]

In [13]:
pattern = r'\b(' + '|'.join(local_markers) + r')\b'
df['code_switched'] = df['clean_review'].str.contains(pattern, case=False, na=False)

coffee_local = df[(df['shop_type'] == 'coffee') & (df['code_switched'] == True)]
coffee_eng = df[(df['shop_type'] == 'coffee') & (df['code_switched'] == False)]
matcha_local = df[(df['shop_type'] == 'matcha') & (df['code_switched'] == True)]
matcha_eng = df[(df['shop_type'] == 'matcha') & (df['code_switched'] == False)]

# Stratified Dynamic Sampling
target_per_type = 250

n_coffee_local = min(125, len(coffee_local))
n_coffee_eng = target_per_type - n_coffee_local
samp_coffee_local = coffee_local.sample(n=n_coffee_local, random_state=42)
samp_coffee_eng = coffee_eng.sample(n=min(n_coffee_eng, len(coffee_eng)), random_state=42)

n_matcha_local = min(125, len(matcha_local))
n_matcha_eng = target_per_type - n_matcha_local
samp_matcha_local = matcha_local.sample(n=n_matcha_local, random_state=42)
samp_matcha_eng = matcha_eng.sample(n=min(n_matcha_eng, len(matcha_eng)), random_state=42)

samples = [samp_coffee_local, samp_coffee_eng, samp_matcha_local, samp_matcha_eng]
stratified_sample = pd.concat(samples).sample(frac=1, random_state=42).reset_index(drop=True)

C:\Users\loren\AppData\Local\Temp\ipykernel_31172\2685591258.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['code_switched'] = df['clean_review'].str.contains(pattern, case=False, na=False)


In [17]:
#sample_path = os.path.join(base_dir, 'taxonomy_sample_500.csv')
#stratified_sample[['place_name', 'shop_type', 'code_switched', 'clean_review']].to_csv(sample_path, index=False, encoding='utf-8-sig')
sample_path = os.path.join(base_dir, 'taxonomy_sample_500.csv')
stratified_sample[['place_name', 'shop_type', 'clean_review']].to_csv(sample_path, index=False, encoding='utf-8-sig')

print(f"Sample generated: {len(stratified_sample)} rows saved to: {sample_path}")
print("Distribution:")
print(stratified_sample.groupby(['shop_type', 'code_switched']).size())

Sample generated: 500 rows saved to: D:/Ateneo de Davao/Thesis-Project-Aspect-Based-Sentiment-Analysis/Dataset\taxonomy_sample_500.csv
Distribution:
shop_type  code_switched
coffee     False            125
           True             125
matcha     False            203
           True              47
dtype: int64
